# Load Libraries

In [31]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Data Preprocessing

# Load Data

In [32]:
# Loading Sensor Data
X = pd.read_csv("secom.data", sep=r"\s+", header=None, na_values="NaN")
X.columns = [f"Sensor_{i:03d}" for i in range(1, 591)]

# Loading Labels: Pass/Fail
L = pd.read_csv("secom_labels.data", sep=r"\s+", header=None,
                names=["label", "timestamp"], quotechar='"')
L["timestamp"] = pd.to_datetime(L["timestamp"], format="%d/%m/%Y %H:%M:%S")
L["label"] = L["label"].map({-1: 0, 1: 1})        # 1 = fail, 0 = pass

assert len(X) == len(L) # Checking that the number of rows in both dataframes is the same
ids = pd.DataFrame({"Unit_ID": range(1, len(X) + 1)}) # Creating a DataFrame for Unit IDs
semcon_df = pd.concat([ids, L, X], axis=1)

semcon_df.to_csv("semcon_df.csv", index=False) # Saving the combined DataFrame to a CSV file

semcon_df.head() # Displaying the first few rows of the combined DataFrame

,Unit_ID,label,timestamp,Sensor_001,Sensor_002,Sensor_003,Sensor_004,Sensor_005,Sensor_006,Sensor_007,...,Sensor_581,Sensor_582,Sensor_583,Sensor_584,Sensor_585,Sensor_586,Sensor_587,Sensor_588,Sensor_589,Sensor_590
0,1,0,2008-07-19 11:55:00,3030.93,2564.00,2187.7333,1411.1265,1.3602,100.0,97.6133,...,NaN,NaN,0.5005,0.0118,0.0035,2.3630,NaN,NaN,NaN,NaN
1,2,0,2008-07-19 12:32:00,3095.78,2465.14,2230.4222,1463.6606,0.8294,100.0,102.3433,...,0.0060,208.2045,0.5019,0.0223,0.0055,4.4447,0.0096,0.0201,0.0060,208.2045
2,3,1,2008-07-19 13:17:00,2932.61,2559.94,2186.4111,1698.0172,1.5102,100.0,95.4878,...,0.0148,82.8602,0.4958,0.0157,0.0039,3.1745,0.0584,0.0484,0.0148,82.8602
3,4,0,2008-07-19 14:43:00,2988.72,2479.90,2199.0333,909.7926,1.3204,100.0,104.2367,...,0.0044,73.8432,0.4990,0.0103,0.0025,2.0544,0.0202,0.0149,0.0044,73.8432
4,5,0,2008-07-19 15:22:00,3032.24,2502.87,2233.3667,1326.5200,1.5334,100.0,100.3967,...,NaN,NaN,0.4800,0.4766,0.1045,99.3032,0.0202,0.0149,0.0044,73.8432


In [33]:
# Displaying basic information about the dataset to verify with README file
print("shape:", semcon_df.shape)
print("fails / passes:", (semcon_df.label == 1).sum(), (semcon_df.label == 0).sum())
print("missing cells:", int(X.isna().sum().sum()), f"({X.isna().mean().mean():.2%})")
print("dates:", semcon_df.timestamp.min(), "to", semcon_df.timestamp.max(),
      "| in order:", semcon_df.timestamp.is_monotonic_increasing)
print(semcon_df.iloc[:3, :6])

shape: (1567, 593)
fails / passes: 104 1463
missing cells: 41951 (4.54%)
dates: 2008-07-19 11:55:00 to 2008-10-17 06:07:00 | in order: True
   Unit_ID  label           timestamp  Sensor_001  Sensor_002  Sensor_003
0        1      0 2008-07-19 11:55:00     3030.93     2564.00   2187.7333
1        2      0 2008-07-19 12:32:00     3095.78     2465.14   2230.4222
2        3      1 2008-07-19 13:17:00     2932.61     2559.94   2186.4111


## Data Verification

In [34]:
sensors = list(X.columns)                             # list of all 590 sensor names

print("Dataset rows:", len(semcon_df), "| Sensor columns:", len(sensors))   
print("Duplicate rows:", X.duplicated().sum(),               
      "| repeated timestamps:", semcon_df["timestamp"].duplicated().sum()) 

Dataset rows: 1567 | Sensor columns: 590
Duplicate rows: 0 | repeated timestamps: 33


## Data Cleaning

In [35]:
# Drop Sensors that are constant or mostly missing
constant = []                                         
for s in sensors:                                     
    if X[s].nunique() <= 1:                          
        constant.append(s)                            
print("constant sensors:", len(constant))            


high_missing = []                                     
for s in sensors:                                     
    if X[s].isna().mean() > 0.5:                      
        high_missing.append(s)                        
print("more than 50% missing:", len(high_missing))    

# Keep everything else
keep = []                                             
for s in sensors:                                     
    if s not in constant and s not in high_missing:   
        keep.append(s)                                
print("sensors left:", len(keep))                     

# Build the clean table
semcon_df = semcon_df[["Unit_ID", "label", "timestamp"] + keep].copy()  
print("clean table:", semcon_df.shape)

constant sensors: 116
more than 50% missing: 28
sensors left: 446
clean table: (1567, 449)


## Split Data into Train/Test

In [36]:
# Split into train and test sets
y = semcon_df["label"]                                  
X_all = semcon_df[keep]                                 
X_train, X_test, y_train, y_test = train_test_split(          
    X_all, y,                                                 
    test_size=0.2,                                            
    stratify=y,                                               
    random_state=42)                                          # fixed seed so everyone gets the same split
print("train:", X_train.shape, "fails:", y_train.sum())       
print("test:", X_test.shape, "fails:", y_test.sum()) 

train: (1253, 446) fails: 83
test: (314, 446) fails: 21


In [37]:
medians = X_train.median()                                    # middle value of each sensor, from train rows only
X_train = X_train.fillna(medians)                             # fill train blanks with those medians
X_test = X_test.fillna(medians)                               # fill test blanks with the same train medians
print("blanks left:", X_train.isna().sum().sum(), X_test.isna().sum().sum())   # check that all blanks are filled

blanks left: 0 0


In [39]:
# Standardize the data
means = X_train.mean()                                        
stds = X_train.std()                                          
X_train = (X_train - means) / stds                            
X_test = (X_test - means) / stds

# Check the result
print("final shape:", X_train.shape, X_test.shape)            
print("blanks:", X_train.isna().sum().sum(), X_test.isna().sum().sum())   
print("train average range:", round(X_train.mean().abs().max(), 4))       
print("train spread range:", round(X_train.std().min(), 3), round(X_train.std().max(), 3))  
print("test average range:", round(X_test.mean().abs().max(), 3))   
print("test spread range:", round(X_test.std().min(), 3), round(X_test.std().max(), 3))   

final shape: (1253, 446) (314, 446)
blanks: 0 0
train average range: 0.0
train spread range: 1.0 1.0
test average range: 8801.227
test spread range: 0.0 6898.99
